# Disaster damage demo: can a written claim sway a vision model?

This notebook demonstrates the code in this repository on a fixed sample of 12 xBD buildings. Each example has a before and after image, an xBD damage label, and four prompts for Qwen3-VL: images only, a matching claim, and two conflicting claims. The question and images stay fixed within each building.

Run the cells from top to bottom. The first sections work with prepared local data and saved predictions. The model cell is opt in because the 8B model needs an accelerator, a large download, and substantial memory.

## Setup

Use Python 3.10 or newer. From the repository root, install and open Jupyter:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[notebook]'
jupyter lab notebooks/disaster_demo.ipynb
```

Select the `.venv` Python kernel. To run Qwen from this notebook, install `python -m pip install -e '.[notebook,demo]'` in that environment. A fresh clone also needs the labeled xBD training data from [xView2](https://xview2.org/dataset). Put its `images/` and `labels/` under `data/raw/train/`, or change `XBD_ROOT` below. The repository intentionally does not track the dataset, model weights, or generated predictions.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
from collections import Counter

from IPython.display import Image, Markdown, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src/disaster_model").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the disaster-model repository.")
sys.path.insert(0, str(ROOT / "src"))

from disaster_model.conditions import ANSWER_CHOICES, demo_conditions
from disaster_model.data import read_manifest
from disaster_model.xbd import prepare_sample

XBD_ROOT = ROOT / "data/raw/train"
MANIFEST = ROOT / "data/processed/manifest.jsonl"
RESULTS = ROOT / "results/demo"
PREDICTIONS = RESULTS / "predictions.jsonl"

print(f"Repository: {ROOT}")
print(f"Prepared sample: {MANIFEST.exists()}")
print(f"Completed run files: {PREDICTIONS.exists()}")

## 1. Prepare or load the 12 buildings

The preparation code selects three buildings per damage class and saves paired 256-pixel crops with a yellow box around the target building. If you already have `data/processed/manifest.jsonl`, this cell just loads it. The selection uses seed 345; inspect the crops before presenting a new sample. See [the runbook](../docs/midterm-demo.md) for the review steps.

In [ ]:
if not MANIFEST.exists() and XBD_ROOT.is_dir():
    MANIFEST = prepare_sample(XBD_ROOT, ROOT / "data/processed", per_class=3, crop_size=256, seed=345)
    print(f"Prepared sample: {MANIFEST}")
elif not MANIFEST.exists():
    print("No prepared sample found. Download xBD, set XBD_ROOT above, and rerun this cell.")

buildings = list(read_manifest(MANIFEST)) if MANIFEST.exists() else []
if buildings:
    missing = [
        building.building_id
        for building in buildings
        if not building.pre_image.is_file() or not building.post_image.is_file()
    ]
    if missing:
        raise FileNotFoundError(f"Missing crops for {len(missing)} buildings; prepare the sample again.")
    labels = Counter(building.true_label for building in buildings)
    print(f"Loaded {len(buildings)} buildings: {dict(labels)}")

## 2. Inspect the model input

The same two crops are used in all four conditions for a building. The yellow box identifies the target; it does not encode its damage label.

In [ ]:
if buildings:
    building = buildings[0]
    print(f"Building: {building.building_id}")
    print(f"xBD label: {building.true_label}")
    display(Markdown("**Before disaster**"))
    display(Image(filename=str(building.pre_image)))
    display(Markdown("**After disaster**"))
    display(Image(filename=str(building.post_image)))
else:
    print("The image pair appears after the xBD sample is prepared.")

## 3. Inspect the four prompts

The A/B/C/D choices are fixed. For each building, the claims are selected from the xBD label: one matching claim, the nearest conflicting class, and the farthest conflicting class. This is a small paired demonstration, not a random test set.

In [ ]:
print("Answer choices:", ANSWER_CHOICES)
example_label = buildings[0].true_label if buildings else "no damage"
for condition in demo_conditions(example_label):
    print(f"\n[{condition.name}; claim={condition.text_claim!r}; gap={condition.conflict_distance}]")
    print(condition.prompt)

## 4. Optionally run Qwen3-VL

Set `RUN_INFERENCE = True` to run one pilot prompt and then all 48 prompts. The pilot checks the A/B/C/D answer tokens. The full run saves `predictions.jsonl` and `run.json` under `RESULTS`. It refuses to overwrite existing predictions; choose a new `RESULTS` directory for a new run. The first run downloads model weights. A GPU or Apple Metal accelerator is expected.

In [ ]:
RUN_INFERENCE = False

if RUN_INFERENCE:
    if not buildings:
        raise RuntimeError("Prepare the xBD sample before running inference.")
    if PREDICTIONS.exists():
        raise FileExistsError(f"{PREDICTIONS} exists. Set RESULTS to a new directory and rerun the setup cell.")
    common = [
        sys.executable, str(ROOT / "scripts/run_demo.py"),
        "--manifest", str(MANIFEST),
        "--output", str(RESULTS),
    ]
    subprocess.run([*common, "--pilot-only"], cwd=ROOT, check=True)
    subprocess.run(common, cwd=ROOT, check=True)
else:
    print("Using saved predictions when available. Set RUN_INFERENCE = True to run Qwen.")

## 5. Summarize the saved run

The scorer takes the next-token logits for A, B, C, and D, then applies softmax over those four values. Each row's shares sum to 1. They express the model's preference *among these choices*; they are not calibrated probabilities that the answer is correct. We also check that the run is complete and corresponds to this exact manifest.

In [ ]:
rows = []
run = None
if PREDICTIONS.is_file() and MANIFEST.is_file():
    run = json.loads((RESULTS / "run.json").read_text(encoding="utf-8"))
    if not run.get("complete"):
        raise ValueError("The inference run is incomplete.")
    digest = hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
    if run["manifest_sha256"] != digest:
        raise ValueError("These predictions belong to a different manifest.")
    rows = [
        json.loads(line) for line in PREDICTIONS.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    expected = {building.building_id for building in buildings}
    actual = {row["building_id"] for row in rows}
    if actual != expected or len(rows) != 4 * len(buildings):
        raise ValueError("Saved predictions do not cover exactly four prompts per building.")
    print(f"Model: {run['model_id']}")
    print(f"Revision: {run['model_revision']}")
    print(f"Device: {run['device']}; prompts: {len(rows)}")
else:
    print("No saved run is available. Complete the sample and inference steps above.")

In [ ]:
if rows:
    baseline = [row for row in rows if row["condition"] == "image_only"]
    conflicts = [row for row in rows if row["condition"] == "conflicting_text"]
    eligible = [
        row for row in conflicts
        if row["image_only_prediction"] != row["text_claim"]
    ]
    correct = sum(row["predicted_label"] == row["true_label"] for row in baseline)
    followed = sum(row["predicted_label"] == row["text_claim"] for row in conflicts)
    flipped = sum(row["changed_toward_text"] for row in eligible)
    display(Markdown(
        f"**Image-only agreement with xBD:** {correct}/{len(baseline)}  \n"
        f"**Conflicting claims followed:** {followed}/{len(conflicts)}  \n"
        f"**Eligible conflicts that moved toward the claim:** {flipped}/{len(eligible)}"
    ))
    print("Eligible means the claim differs from the image-only prediction.")
else:
    print("Summary appears after inference.")

## 6. Compare one building across conditions

The table shows the same building's image-only answer, matching claim, and two conflicting claims. “Share” is the winning choice's softmax share for that prompt.

In [ ]:
if rows and buildings:
    example = [row for row in rows if row["building_id"] == buildings[0].building_id]
    lines = [
        "| Condition | Claim | Predicted answer | Share | Changed from images only? |",
        "|---|---|---|---:|---|",
    ]
    for row in example:
        claim = row["text_claim"] or "—"
        changed = "—" if row["prediction_changed"] is None else (
            "Yes" if row["prediction_changed"] else "No"
        )
        lines.append(
            f"| {row['condition']} | {claim} | {row['predicted_choice']}: "
            f"{row['predicted_label']} | {row['confidence']:.3f} | {changed} |"
        )
    display(Markdown("\n".join(lines)))
    display(Markdown("**A/B/C/D shares for the image-only prompt**"))
    image_only = next(row for row in example if row["condition"] == "image_only")
    for choice, label in ANSWER_CHOICES.items():
        print(f"{choice} ({label}): {image_only['choice_probabilities'][choice]:.6f}")
else:
    print("The comparison appears after inference.")

## Interpretation

This notebook demonstrates the data path, prompts, model scoring, and paired comparison. The 12 selected buildings are too few for a general accuracy or causal claim. In a talk, explain that an “eligible flip” requires the conflicting claim to differ from the image-only prediction; otherwise the model can follow the claim without changing its answer. The [offline HTML showcase](../results/demo/showcase.html) can be generated with `python scripts/build_showcase.py` after a completed run.